In [ ]:
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6 huggingface_hub diffusers

In [ ]:
# Imports

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import AutoPipelineForText2Image
from datasets import load_dataset
from IPython.display import display, Audio

In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

In [ ]:
# 0. Pick device
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Input: Incoming article or announcement

article_text = """
OpenAI and its partner Microsoft announced a new suite of open research initiatives
aimed at sustainable artificial intelligence and energy-efficient data centers in Europe.
The initiative, led by top computer scientists, promises to reduce power consumption by
40 percent while accelerating training for next-generation foundation models.
Community leaders praised the environmental commitment, although critics noted that
massive infrastructure scaling will still require substantial energy grid investments.
"""

In [ ]:
# 1. Topic Classification

classifier = pipeline("zero-shot-classification", device=device)
topic_result = classifier(article_text, candidate_labels=["Green Tech", "Finance", "Gaming"])
topic = topic_result["labels"][0]


In [ ]:
# 2. Sentiment
sentiment_analyzer = pipeline("sentiment-analysis", device=device)
sentiment = sentiment_analyzer(article_text[:512])[0]

In [ ]:
# 3. Named Entity Recognition
ner = pipeline("ner", device=device)

entities = []
for item in ner(article_text):
  if item["entity"].endswith(("PER", "ORG", "LOC")):
    entities.append(item["word"])

entitites = set(entities)

In [ ]:
# 4. Question Answering
qa = pipeline("question-answering", device=device)
question = "How much will power consumption be reduced?"
answer = qa(question=question, context=article_text)

In [ ]:
# 5. Summarization
summarizer = pipeline("summarization", device=device)
summary = summarizer(article_text, max_length=45, min_length=20, do_sample=False)[0]["summary_text"].strip()

In [ ]:
# 6. Translation
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device=device)
spanish_summary = translator(summary)[0]["translation_text"]

In [ ]:
# 7. Image Generation
image_pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    variant="fp16" if device == "cuda" else None
).to(device)
image_prompt = f"Futuristic green data center, digital art, {topic}"
cover_image = image_pipe(prompt=image_prompt, num_inference_steps=4, guidance_scale=0.0).images[0]

In [ ]:
# 8. Text-to-Speech
tts = pipeline("text-to-speech", model="microsoft/speecht5_tts", device=device)
embeddings = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings[7306]["xvector"]).unsqueeze(0)
speech = tts(summary, forward_params={"speaker_embeddings": speaker_embedding})


In [ ]:
# Display outputs
print("Topic:", topic)
print("Sentiment:", sentiment["label"], sentiment["score"])
print("Entities:", list(entities)[:5])
print("Question:", question,"\n","Answer:", answer["answer"])
print("Summary:", summary)
print("Spanish Summary:", spanish_summary)
display(cover_image)
display(Audio(speech["audio"], rate=speech["sampling_rate"]))
